## Make src/ importable and load project modules

In [10]:
# The notebook and code are in different directories inside the main project dir.
# Hence, we have to inform python where to find it.
# We add the project's src/ folder to the import path.
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

# Import project modules
import config
import cleaning

# Import other external libraries
import pandas as pd
import json

print("Imports OK. Project root:", config.PROJECT_ROOT)

Imports OK. Project root: /home/koala/lab/adversec


## Load the raw data

In [2]:
# Call the loader.
# It walks the six files in config.RAW_FILES, tags each row with its true class, and stacks them into a DF.
raw = cleaning.load_raw_data()

# Show the overall shape
print("\nCombined shape:", raw.shape)

# Peak at the first few raws
raw.head()

 loaded decimal_benign.csv                         rows=1,223,737
 loaded decimal_DoS.csv                            rows=   74,663
 loaded decimal_spoofing-GAS.csv                   rows=    9,991
 loaded decimal_spoofing-RPM.csv                   rows=   54,900
 loaded decimal_spoofing-SPEED.csv                 rows=   24,951
 loaded decimal_spoofing-STEERING_WHEEL.csv        rows=   19,977

Combined shape: (1408219, 13)


,ID,DATA_0,DATA_1,DATA_2,DATA_3,DATA_4,DATA_5,DATA_6,DATA_7,label,category,specific_class,true_class
0,65,96,0,0,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
1,1068,132,13,160,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
2,535,127,255,127,255,127,255,127,255,BENIGN,BENIGN,BENIGN,benign
3,131,15,224,0,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
4,936,1,0,39,16,0,0,0,0,BENIGN,BENIGN,BENIGN,benign


## Audit the raw duplication

In [3]:
# The signature columns: the 9 CAN features plus the true class.
# Two rows are "the same" only if all nine values AND the class match.
signature_cols = config.FEATURE_COLUMNS + ["true_class"]

# Run the audit over the raw data
raw_audit = cleaning.audit_duplication(raw, subset=signature_cols)

# Print the findings
print("RAW DUPLICATION AUDIT")
print(f"    total rows          : {raw_audit['total_rows']:>10,}")
print(f"    unique signatures   : {raw_audit['unique_signatures']:>10,}")
print(f"    duplicate rows      : {raw_audit['duplicate_rows']:>10,}")
print(f"    duplication rate    : {raw_audit['duplication_rate_pct']:.4f}%")

RAW DUPLICATION AUDIT
    total rows          :  1,408,219
    unique signatures   :      3,588
    duplicate rows      :  1,404,631
    duplication rate    : 99.7452%


## Strict de-duplication

In [4]:
# Run strict de-dup: keep one row per unique (features + class) signature
strict = cleaning.strict_dedup(raw, config.FEATURE_COLUMNS)

print("STRICT DE-DUPLICATION")
print(f"    rows before     : {len(raw):>10,}")
print(f"    rows after      : {len(strict):>10,}")
print(f"    kept            : {len(strict) / len(raw) * 100:.4f}% of original\n")

# The critical view: how many unique signatures per class.
# value_counts() tallies each class; we sort by index so the classes read in a stable order.
print("Unique signatures per class")
print(strict["true_class"].value_counts().sort_index())

STRICT DE-DUPLICATION
    rows before     :  1,408,219
    rows after      :      3,588
    kept            : 0.2548% of original

Unique signatures per class
true_class
DoS                          21
benign                     3547
spoofing-GAS                  2
spoofing-RPM                 10
spoofing-SPEED                5
spoofing-STEERING_WHEEL       3
Name: count, dtype: int64


## Stage 1 Summary: The Accuracy Trap, Confirmed

This notebook loads, audits, and de-duplicates the CICIoV2024 decimal dataset.

### What we found

| Metric | Value |
|---|---|
| Total raw rows | 1,408,219 |
| Unique signatures (features + class) | **3,588** |
| Duplicate rows | 1,404,631 |
| Duplication rate | **99.75%** |

The raw dataset is 99.75% redundant copies. This independently reproduces the
finding of Le & Alsmadi (2026) on our own copy of the data, and is the empirical
basis for the *accuracy trap*: the near-perfect F1 scores reported in prior work
are earned on data where each pattern is repeated tens of thousands of times.

### Unique signatures per class (strict de-duplication)

| Class | Raw rows | Unique signatures |
|---|---|---|
| benign | 1,223,737 | 3,547 |
| DoS | 74,663 | 21 |
| spoofing-RPM | 54,900 | 10 |
| spoofing-SPEED | 24,951 | 5 |
| spoofing-STEERING_WHEEL | 19,977 | 3 |
| spoofing-GAS | 9,991 | 2 |
| **Total** | **1,408,219** | **3,588** |

Only **41 unique attack signatures** exist across all five attack types. The GAS
attack is just 2 distinct frames repeated roughly 10,000 times. A model scoring a
perfect F1 here has memorised a handful of patterns, not learned to detect
intrusions.

### Implication for the experiment

The strict set cannot be trained honestly as a six-class problem: a class of 2–5
examples is not learnable. Naive duplication (repeating those few frames) would
only recreate the accuracy trap. The augmentation strategy is therefore:

- **Strict set** → truth/analysis set; exposes the trap.
- **Light duplication** → solves convergence (gives the optimiser enough volume
  to train a baseline and form gradients).
- **Adversarial synthesis (FGSM/PGD)** → solves diversity (generates new
  near-miss samples around the real signatures).

Adversarial samples will be generated from **training-fold signatures only**,
with the test set held as **real signatures**, to prevent the trap returning in
disguise through train/test leakage.

## Split the strict set into train and test

In [5]:
# Split the strict unique signatures into train and test before augmentation.
# Classes with <6 signatures send exactly 1 to test. Larger classes send 20%
train, test = cleaning.split_train_test(strict, config.FEATURE_COLUMNS)

print("SPLIT RESULT")
print(f"    train rows  : {len(train):>6,}")
print(f"    test rows   : {len(test):>6,}\n")

# Per-class breakdown for both splits, side by side
split_summary = pd.DataFrame({
    "train": train["true_class"].value_counts(),
    "test": test["true_class"].value_counts(),
}).fillna(0).astype(int).sort_index()

print(split_summary)

SPLIT RESULT
    train rows  :  2,870
    test rows   :    718

                         train  test
true_class                          
DoS                         17     4
benign                    2838   709
spoofing-GAS                 1     1
spoofing-RPM                 8     2
spoofing-SPEED               4     1
spoofing-STEERING_WHEEL      2     1


## Prove there is no train/test leakage

In [6]:
# Prove no signature appears in both train and test.
# Build a "signature key" for every row (the 9 features + class joined into one string).
# Then check the overlap between the two sets.
# A leakage-free split should have zero overlaps.

sig_cols = config.FEATURE_COLUMNS + ["true_class"]

def signature_key(df):
    # Turn each row's signature columns into a single tuple, then into a set.
    return set(map(tuple, df[sig_cols].itertuples(index=False, name=None)))

train_keys = signature_key(train)
test_keys = signature_key(test)

# The intersection is the set of signatures present in both.
overlap = train_keys & test_keys

print("LEAKAGE CHECK")
print(f"    unique train signatures     : {len(train_keys):>6,}")
print(f"    unique test signatures      : {len(test_keys):>6,}")
print(f"    overlap (must be 0)         : {len(overlap)}")
print()
print("PASS: no leakage" if len(overlap) == 0 else "FAIL: leakage detected")

LEAKAGE CHECK
    unique train signatures     :  2,870
    unique test signatures      :    718
    overlap (must be 0)         : 0

PASS: no leakage


## Light duplication on the train split only

In [7]:
# Apply light duplication to train split only.
# Attack classes are duplicated up to 200 rows each while benign is left untouched.

train_dup = cleaning.duplicate_train_classes(train, target_per_class=200)

print("TRAIN AFTER LIGHT DUPLICATION")
print(f"    rows before     : {len(train):>6,}")
print(f"    rows after      : {len(train_dup):>6,}")

# Per-class counts after duplication
print("Per-class counts (train): ")
print(train_dup["true_class"].value_counts().sort_index())

TRAIN AFTER LIGHT DUPLICATION
    rows before     :  2,870
    rows after      :  3,838
Per-class counts (train): 
true_class
DoS                         200
benign                     2838
spoofing-GAS                200
spoofing-RPM                200
spoofing-SPEED              200
spoofing-STEERING_WHEEL     200
Name: count, dtype: int64


## Re-prove no leakage after duplication

In [9]:
# Re-check leakage between the duplicated train set and the frozen test set.

train_dup_keys = signature_key(train_dup)
test_keys = signature_key(test)

overlap_after = train_dup_keys & test_keys

print("POST-DUPLICATION LEAKAGE CHECK")
print(f"    unique train signatures     : {len(train_dup_keys):>6,}")
print(f"    unique test signatures      : {len(test_keys):>6,}")
print(f"    overlap (must be 0)         : {len(overlap_after)}")
print()
print("PASS: test still clean" if len(overlap_after) == 0 else "FAIL: leakage detected")

POST-DUPLICATION LEAKAGE CHECK
    unique train signatures     :  2,870
    unique test signatures      :    718
    overlap (must be 0)         : 0

PASS: test still clean


## Save processed datasets and audit report

In [13]:
# Make sure output folders exist
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# The columns to keep in the saved files: 9 features plus true class.
keep_cols = config.FEATURE_COLUMNS + ["true_class"]

# Save 3 datasets
strict_path = config.PROCESSED_DIR / "ciciov2024_strict.csv"
train_path = config.PROCESSED_DIR / "ciciov2024_train_dup.csv"
test_path = config.PROCESSED_DIR / "ciciov2024_test.csv"

strict[keep_cols].to_csv(strict_path, index=False)
train_dup[keep_cols].to_csv(train_path, index=False)
test[keep_cols].to_csv(test_path, index=False)

# Build the audit report
audit = {
    "dataset": "CICIoV2024 (decimal)",
    "random_seed": config.RANDOM_SEED,
    "raw": raw_audit,  # from Cell 3
    "strict_total": int(len(strict)),
    "strict_per_class": {k: int(v) for k, v in strict["true_class"].value_counts().sort_index().items()},
    "split_rule": "test = 20% for classes with >=6 signatures, else 1-signature floor",
    "train_per_class_real": {k: int(v) for k, v in train["true_class"].value_counts().sort_index().items()},
    "test_per_class": {k: int(v) for k, v in test["true_class"].value_counts().sort_index().items()},
    "train_duplication_target": 200,
    "train_per_class_after_dup": {k: int(v) for k, v in train_dup["true_class"].value_counts().sort_index().items()},
    "leakage_overlap": int(len(overlap_after)),
}

audit_path = config.RESULTS_DIR / "stage1_audit_report.json"
with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print("Saved")
for p in (strict_path, train_path, test_path, audit_path):
    print(" ", p)

Saved
  /home/koala/lab/adversec/datasets/processed/ciciov2024_strict.csv
  /home/koala/lab/adversec/datasets/processed/ciciov2024_train_dup.csv
  /home/koala/lab/adversec/datasets/processed/ciciov2024_test.csv
  /home/koala/lab/adversec/results/stage1_audit_report.json
